# Enhanced CPS 2024 Data Exploration

This notebook explores the enhanced_cps_2024 dataset to understand its structure, demographics, and income distribution.

In [1]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np

ENHANCED_CPS = "hf://policyengine/policyengine-us-data/enhanced_cps_2024.h5"

## 1. Dataset Overview

In [2]:
# Load enhanced_cps_2024 dataset
sim = Microsimulation(dataset=ENHANCED_CPS)

In [3]:
# Check dataset size
household_weight = sim.calc("household_weight", period=2026)
household_count = sim.calc("household_count", period=2026, map_to="household")
person_count = sim.calc("person_count", period=2026, map_to="household")

print(f"Number of households in dataset: {len(household_weight):,}")
print(f"Household count (weighted): {household_count.sum():,.0f}")
print(f"Person count (weighted): {person_count.sum():,.0f}")

Number of households in dataset: 15,929
Household count (weighted): 144,287,680
Person count (weighted): 339,597,307


## 2. Income Distribution

In [4]:
# Check household income distribution
agi = sim.calc("adjusted_gross_income", period=2026, map_to="household")
print(f"Income distribution:")
print(f"  Median AGI: ${agi.median():,.0f}")
print(f"  75th percentile: ${agi.quantile(0.75):,.0f}")
print(f"  90th percentile: ${agi.quantile(0.90):,.0f}")
print(f"  95th percentile: ${agi.quantile(0.95):,.0f}")
print(f"  Max AGI: ${agi.max():,.0f}")

Income distribution:
  Median AGI: $54,423
  75th percentile: $117,905
  90th percentile: $202,534
  95th percentile: $324,016
  Max AGI: $349,003,552


In [5]:
# Employment income analysis
employment_income = sim.calc("employment_income", period=2026, map_to="person")
employment_income_before_lsr = sim.calc("employment_income_before_lsr", period=2026, map_to="person")

print(f"\nEmployment Income Analysis:")
print(f"  Total employment income (weighted): ${employment_income.sum():,.0f}")
print(f"  Total employment income before LSR (weighted): ${employment_income_before_lsr.sum():,.0f}")
print(f"  Mean employment income: ${employment_income.mean():,.0f}")
print(f"  Mean employment income before LSR: ${employment_income_before_lsr.mean():,.0f}")
print(f"  Median employment income: ${employment_income.median():,.0f}")
print(f"  Median employment income before LSR: ${employment_income_before_lsr.median():,.0f}")


Employment Income Analysis:
  Total employment income (weighted): $10,943,358,507,618
  Total employment income before LSR (weighted): $10,943,358,507,618
  Mean employment income: $32,225
  Mean employment income before LSR: $32,225
  Median employment income: $0
  Median employment income before LSR: $0


In [6]:
# Household counts by income brackets
agi_hh = np.array(sim.calc("adjusted_gross_income", period=2026, map_to="household").values)
weights = np.array(sim.calc("household_weight", period=2026).values)
total_households = weights.sum()

income_brackets = [
    (0, 10000, "$0-$10k"),
    (10000, 25000, "$10k-$25k"),
    (25000, 50000, "$25k-$50k"),
    (50000, 75000, "$50k-$75k"),
    (75000, 100000, "$75k-$100k"),
    (100000, 150000, "$100k-$150k"),
    (150000, 200000, "$150k-$200k"),
    (200000, float('inf'), "$200k+")
]

bracket_data = []
for lower, upper, label in income_brackets:
    mask = (agi_hh >= lower) & (agi_hh < upper)
    count = weights[mask].sum()
    pct_of_total = (count / total_households) * 100
    
    bracket_data.append({
        "Income Bracket": label,
        "Households": f"{count:,.0f}",
        "% of All Households": f"{pct_of_total:.2f}%"
    })

income_df = pd.DataFrame(bracket_data)

print("\n" + "="*70)
print("HOUSEHOLD COUNTS BY INCOME BRACKET")
print("="*70)
print(income_df.to_string(index=False))
print("="*70)


HOUSEHOLD COUNTS BY INCOME BRACKET
Income Bracket Households % of All Households
       $0-$10k 28,130,220              19.50%
     $10k-$25k 15,020,060              10.41%
     $25k-$50k 24,209,866              16.78%
     $50k-$75k 17,464,564              12.10%
    $75k-$100k 15,011,586              10.40%
   $100k-$150k 18,161,620              12.59%
   $150k-$200k 10,102,187               7.00%
        $200k+ 14,839,484              10.28%


## 3. Household Composition

In [7]:
# Check households with children
is_child = sim.calc("is_child", period=2026, map_to="person")
household_id = sim.calc("household_id", period=2026, map_to="person")
person_weight = sim.calc("person_weight", period=2026, map_to="person")
household_weight_person = sim.calc("household_weight", period=2026, map_to="person")

# Create DataFrame
df_households = pd.DataFrame({
    'household_id': household_id.values,
    'is_child': is_child.values,
    'household_weight': household_weight_person.values
})

# Count children per household
children_per_household = df_households.groupby('household_id').agg({
    'is_child': 'sum',
    'household_weight': 'first'
}).reset_index()

# Calculate weighted household counts
total_households_with_children = children_per_household[children_per_household['is_child'] > 0]['household_weight'].sum()
households_with_1_child = children_per_household[children_per_household['is_child'] == 1]['household_weight'].sum()
households_with_2_children = children_per_household[children_per_household['is_child'] == 2]['household_weight'].sum()
households_with_3plus_children = children_per_household[children_per_household['is_child'] >= 3]['household_weight'].sum()

print(f"\nHouseholds with children (weighted):")
print(f"  Total households with children: {total_households_with_children:,.0f}")
print(f"  Households with 1 child: {households_with_1_child:,.0f}")
print(f"  Households with 2 children: {households_with_2_children:,.0f}")
print(f"  Households with 3+ children: {households_with_3plus_children:,.0f}")


Households with children (weighted):
  Total households with children: 43,798,516
  Households with 1 child: 24,479,972
  Households with 2 children: 12,298,578
  Households with 3+ children: 7,019,965


In [8]:
# Check children by age groups
age = sim.calc("age", period=2026, map_to="person")

df_age = pd.DataFrame({
    "age": age.values,
    "person_weight": person_weight.values
})

# Calculate weighted totals
total_children = df_age[df_age['age'] < 18]['person_weight'].sum()
children_under_6 = df_age[df_age['age'] < 6]['person_weight'].sum()
children_under_3 = df_age[df_age['age'] < 3]['person_weight'].sum()

print(f"\nChildren by age:")
print(f"  Total children under 18: {total_children:,.0f}")
print(f"  Children under 6: {children_under_6:,.0f}")
print(f"  Children under 3: {children_under_3:,.0f}")


Children by age:
  Total children under 18: 73,113,736
  Children under 6: 22,428,570
  Children under 3: 11,172,289
